# Semana 9: RQ4, exportação das imagens Sentinel-2 do Rio Grande do Sul

Seção 1 de `prereg/plano_rq4.md` (commit `245f811`). **Este notebook não roda nenhum modelo e não lê nenhuma referência de enchente.** Ele só pede ao Google Earth Engine as imagens Sentinel-2 L1C de **2024-05-06 (UTC)** dos tiles **22JDM** (Porto Alegre e Canoas), **22JDN** (vale do Taquari: Encantado, Roca Sales; e São Leopoldo) e **22JCN** (baixo Taquari e Jacuí: Lajeado, Rio Pardo) e as guarda no Drive, com o registro do que foi baixado. O 22JDN entrou em 2026-11-02, quando vi que Encantado fica nele e não no 22JCN (registro no fim de `prereg/plano_rq4.md`).

**Segunda rodada.** O que já foi exportado e colocado na grade (22JDM e 22JCN) não é exportado nem regravado de novo: a célula 5 só confere que os SHA-256 desses arquivos são os mesmos registrados no repositório (`analysis/rq4/entradas/rq4_entradas.json`). A saída de cada rodada vai para uma pasta nova (`rq4_entradas`, depois `rq4_entradas_2`, e assim por diante); a célula 1 mostra qual.

Para cada tile são exportados três arquivos na grade de 10 m do próprio granulo:

| Arquivo | Coleção | Conteúdo |
|---|---|---|
| `..._s2.tif` | `COPERNICUS/S2_HARMONIZED` | B2, B3, B4, B8A, B11, B12, uint16. A coleção harmonizada tira o deslocamento de 1.000 que as imagens L1C passam a ter a partir da linha de base 04.00, então os valores ficam na escala dos dados de treino. Pixel sem dado = 0. B8A, B11 e B12 são de 20 m e entram por vizinho mais próximo (padrão do Earth Engine). |
| `..._nuvem.tif` | `COPERNICUS/S2_CLOUD_PROBABILITY` | `probability` do s2cloudless, uint8. Sem dado = 255 (nunca lido como céu limpo). |
| `..._scl.tif` | `COPERNICUS/S2_SR_HARMONIZED` | `SCL` do L2A do mesmo granulo, uint8 (0 = sem dado). |

O Earth Engine corta a exportação na área da imagem (arredondada para fora na grade de 10 m), então o arquivo bruto pode ser maior ou menor que o granulo. A célula 5 o recoloca na grade inteira por deslocamento de pixels inteiros, sem mudar nenhum valor. Se houver mais de um granulo do tile no dia, vale o de menor `CLOUDY_PIXEL_PERCENTAGE`; se nenhum existir, o notebook para (o plano, seção 5, diz o que fazer, e eu decido com você).

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive, clona o repositório, entra no Earth Engine |
| 2 | Lista os granulos de 2024-05-06 de cada tile e escolhe um |
| 3 | Confere que a nuvem e a SCL existem para o mesmo granulo; confere a linha de base e o deslocamento; lê a grade do B2 |
| 4 | Pede ao Earth Engine as exportações que ainda não existem (3 por tile) e espera terminar (de 20 a 60 minutos) |
| 5 | Move os arquivos para `Mestrado/dados/rs2024/s2`, coloca cada um na grade inteira do granulo (10980 × 10980, pasta `canonico`, sem reamostrar), confere a grade, calcula os SHA-256 e grava `rq4_entradas.json` |

**Antes de rodar:** abra o Colab logada na conta do mestrado (não precisa de GPU). O `PROJETO_GEE` da célula 1 já vem preenchido com `mestrado-prithvi`. A célula 1 abre a janela de login do Earth Engine na primeira vez. Se a sessão cair durante a célula 4, rode a célula 1 e a 4 de novo: as exportações já pedidas não são repetidas. Se alguma célula der erro, pare e me mande a saída inteira.

Depois da célula 5, me mande `rq4_entradas.json` e `rq4_entradas_sha256.csv` da pasta de saída desta rodada.

In [ ]:
# Célula 1 de 5: Drive, repositório e Earth Engine
import os, re, json, time, hashlib, glob, shutil, subprocess, sys, csv
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

DESTINO = f'{RAIZ}/dados/rs2024/s2'
PASTA_EXPORT = 'rq4_s2'                      # pasta criada pelo Earth Engine na raiz do seu Drive
# Nunca sobrescreve: se a pasta já tem arquivos de uma rodada anterior, usa a próxima livre (_2, _3, ...).
SAIDA_BASE = f'{RAIZ}/resultados/semana9/rq4_entradas'
SAIDA_DRIVE, n = SAIDA_BASE, 1
while os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE):
    n += 1
    SAIDA_DRIVE = f'{SAIDA_BASE}_{n}'
print('Saída desta rodada:', SAIDA_DRIVE.replace(MEUDRIVE, 'MyDrive'))
os.makedirs(DESTINO, exist_ok=True)

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO_GIT}/analysis/rq4')
import s2_exportacao as S
SHA_MODULO = hashlib.sha256(open(f'{REPO_GIT}/analysis/rq4/s2_exportacao.py', 'rb').read()).hexdigest()

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

import ee
PROJETO_GEE = 'mestrado-prithvi'              # ID do projeto do Earth Engine (já preenchido)
assert PROJETO_GEE, 'Preencha PROJETO_GEE com o ID do projeto do Earth Engine.'
try:
    ee.Initialize(project=PROJETO_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJETO_GEE)

L1C, NUVEM, L2A = 'COPERNICUS/S2_HARMONIZED', 'COPERNICUS/S2_CLOUD_PROBABILITY', 'COPERNICUS/S2_SR_HARMONIZED'
print(f'Repositório no commit {commit[:7]}. Earth Engine pronto (projeto {PROJETO_GEE}).')

In [ ]:
# Célula 2 de 5: escolha dos granulos
PROPS = ['system:index', 'system:time_start', 'MGRS_TILE', 'PRODUCT_ID', 'GRANULE_ID', 'CLOUDY_PIXEL_PERCENTAGE',
         'PROCESSING_BASELINE', 'SPACECRAFT_NAME', 'SENSING_ORBIT_NUMBER']
ESCOLHIDOS, TODOS = {}, {}
for tile in S.TILES:
    col = (ee.ImageCollection(L1C).filter(ee.Filter.eq('MGRS_TILE', tile))
           .filterDate(S.DATA, '2024-05-07'))                 # dia 2024-05-06 em UTC
    feats = col.getInfo()['features']
    cand = [{'id': f['properties']['system:index'], 'tile': f['properties']['MGRS_TILE'],
             'time_start_ms': f['properties']['system:time_start'],
             'nuvem': f['properties']['CLOUDY_PIXEL_PERCENTAGE'], 'n_candidatos': len(feats),
             'propriedades': {p: f['properties'].get(p) for p in PROPS}} for f in feats]
    TODOS[tile] = [{k: v for k, v in c.items() if k != 'propriedades'} for c in cand]
    print(f'{tile}: {len(cand)} granulo(s) em {S.DATA}')
    for c in cand:
        print('   ', c['id'], f"nuvem {c['nuvem']:.1f}%", c['propriedades']['PROCESSING_BASELINE'], c['propriedades']['SPACECRAFT_NAME'])
    escolhido = S.escolher_granulo(cand, tile)
    escolhido['n_candidatos'] = len(feats)
    S.conferir_linha_base(escolhido['propriedades']['PROCESSING_BASELINE'])
    ESCOLHIDOS[tile] = escolhido
    print(f'   escolhido: {escolhido["id"]}')

In [ ]:
# Célula 3 de 5: conferências antes de exportar
COLECOES = {'l1c': L1C, 'nuvem': NUVEM, 'l2a': L2A}
GRADES, DESLOC = {}, {}
for tile, c in ESCOLHIDOS.items():
    idx = c['id']
    for nome, colecao in (('nuvem', NUVEM), ('l2a', L2A)):
        try:
            ee.Image(f'{colecao}/{idx}').getInfo()
        except ee.EEException as erro:
            raise RuntimeError(f'{tile}: não há {nome} ({colecao}) para o granulo {idx}. Pare e me mande esta saída.') from erro
    proj = ee.Image(f'{L1C}/{idx}').select('B2').projection().getInfo()
    a, b, x0, d, e, y0 = proj['transform']
    assert (a, b, d, e) == (10, 0, 0, -10) and proj['crs'] == S.CRS_ESPERADO[tile], (tile, proj)
    GRADES[tile] = {'crs': proj['crs'], 'transform': proj['transform'], 'origem': (float(x0), float(y0))}

    # Conferência do deslocamento: no mesmo local, o valor bruto menos o harmonizado deve ser 1000 (menos onde o
    # harmonizado foi cortado em 0). Se a coleção bruta não estiver mais disponível, isso fica registrado.
    centro = ee.Geometry.Rectangle([x0 + 5000, y0 - 5500, x0 + 6000, y0 - 4500], proj=proj['crs'], geodesic=False)
    harm = ee.Image(f'{L1C}/{idx}').select('B4').reduceRegion(ee.Reducer.mean(), centro, 10).get('B4').getInfo()
    try:
        bruto = ee.Image(f'COPERNICUS/S2/{idx}').select('B4').reduceRegion(ee.Reducer.mean(), centro, 10).get('B4').getInfo()
        dif = float(bruto) - float(harm)
        DESLOC[tile] = {'situacao': 'conferido' if 900 <= dif <= 1000.5 else 'FORA DO ESPERADO',
                        'b4_bruto_menos_harmonizado': dif, 'b4_harmonizado': float(harm)}
    except Exception as erro:
        DESLOC[tile] = {'situacao': 'colecao bruta indisponivel', 'b4_harmonizado': float(harm), 'erro': type(erro).__name__}
    print(tile, GRADES[tile]['crs'], 'origem', GRADES[tile]['origem'], DESLOC[tile])
    assert DESLOC[tile]['situacao'] != 'FORA DO ESPERADO', f'{tile}: o deslocamento não é o esperado. Pare e me mande esta saída.'

In [ ]:
# Célula 4 de 5: exportações
def imagens(tile):
    idx = ESCOLHIDOS[tile]['id']
    s2 = (ee.Image(f'{L1C}/{idx}').select(list(S.BANDAS)).toUint16().unmask(0))
    nuvem = ee.Image(f'{NUVEM}/{idx}').select('probability').toUint8().unmask(255)
    scl = ee.Image(f'{L2A}/{idx}').select('SCL').toUint8().unmask(0)
    return {'s2': s2, 'nuvem': nuvem, 'scl': scl}

def regiao(tile):
    x0, y0 = GRADES[tile]['origem']
    n = S.LADO * S.PIXEL
    return ee.Geometry.Rectangle([x0, y0 - n, x0 + n, y0], proj=GRADES[tile]['crs'], geodesic=False)

ATIVAS = {t.config.get('description'): t for t in ee.batch.Task.list() if t.state in ('READY', 'RUNNING')}
TAREFAS = {}
for tile in S.TILES:
    for nome, img in imagens(tile).items():
        prefixo = f'{tile}_{ESCOLHIDOS[tile]["id"]}_{nome}'
        ja = glob.glob(f'{MEUDRIVE}/{PASTA_EXPORT}/{prefixo}*.tif') + glob.glob(f'{DESTINO}/{prefixo}*.tif')
        if ja:
            print(f'{prefixo}: já exportado ({len(ja)} arquivo).')
            continue
        descricao = prefixo[:100]
        if descricao in ATIVAS:
            TAREFAS[prefixo] = ATIVAS[descricao]
            print(f'{prefixo}: tarefa já em andamento, não repito.')
            continue
        tarefa = ee.batch.Export.image.toDrive(
            image=img, description=descricao, folder=PASTA_EXPORT, fileNamePrefix=prefixo,
            region=regiao(tile), crs=GRADES[tile]['crs'], crsTransform=GRADES[tile]['transform'],
            maxPixels=int(1e10), fileFormat='GeoTIFF')
        tarefa.start()
        TAREFAS[prefixo] = tarefa
        print(f'{prefixo}: tarefa pedida.')

limite = time.time() + 3 * 3600
ultimo = None
while TAREFAS and time.time() < limite:
    estados = {p: t.status()['state'] for p, t in TAREFAS.items()}
    if estados != ultimo:
        print(time.strftime('%H:%M:%S'), {p.split('_')[-1] + '_' + p[:5]: s for p, s in estados.items()})
        ultimo = estados
    ruins = {p: s for p, s in estados.items() if s in ('FAILED', 'CANCELLED', 'CANCEL_REQUESTED')}
    assert not ruins, f'Exportação falhou: {ruins}. Pare e me mande esta saída.'
    if all(s == 'COMPLETED' for s in estados.values()):
        break
    time.sleep(60)
else:
    if TAREFAS:
        raise RuntimeError('As exportações não terminaram em 3 horas. Rode esta célula de novo mais tarde.')
print('Exportações concluídas.')

In [ ]:
# Célula 5 de 5: grade inteira, conferência e registro
import rasterio

def esperar(caminho, segundos=180):
    # O Drive montado no Colab às vezes demora a mostrar um arquivo recém-gravado ou movido.
    os.sync()
    for _ in range(segundos // 5):
        if os.path.exists(caminho):
            return caminho
        time.sleep(5)
    pasta = os.path.dirname(caminho)
    raise FileNotFoundError(f'{caminho} não apareceu em {segundos} s. Pasta: {sorted(os.listdir(pasta))[:40] if os.path.isdir(pasta) else "não existe"}')

CANON = f'{DESTINO}/canonico'
os.makedirs(CANON, exist_ok=True)
# Os arquivos da grade inteira são montados no disco local do Colab e só depois copiados para o Drive, que
# às vezes perde o arquivo logo depois de um rename. A cópia no Drive é conferida pelo SHA-256 no fim de cada arquivo.
LOCAL = '/content/rq4_canonico'
os.makedirs(LOCAL, exist_ok=True)
PREENCHE = {'s2': 0, 'nuvem': 255, 'scl': 0}
ENTRADAS, LINHAS, REPAROS = [], [], []
# O que já está registrado no repositório (rodadas anteriores). O arquivo bruto do Earth Engine é a âncora: tem de
# ter o mesmo SHA-256. O arquivo da grade inteira (canônico) é derivado do bruto, sem reamostrar, e pode ser refeito.
REGISTRO = json.load(open(f'{REPO_GIT}/analysis/rq4/entradas/rq4_entradas.json'))
antigos = {(e['tile'], nome, tipo): e['arquivos'][nome][tipo]['sha256']
           for e in REGISTRO['entradas'] for nome in e['arquivos'] for tipo in ('bruto', 'canonico')}
ZEROS_REGISTRADOS = {e['tile']: e['fracao_pixels_com_6_bandas_zero'] for e in REGISTRO['entradas']}
for tile, c in ESCOLHIDOS.items():
    arquivos, grade, zeros, cobertura = {}, None, None, None
    for nome in ('s2', 'nuvem', 'scl'):
        prefixo = f'{tile}_{c["id"]}_{nome}'
        origem = glob.glob(f'{MEUDRIVE}/{PASTA_EXPORT}/{prefixo}*.tif')
        destino = glob.glob(f'{DESTINO}/{prefixo}*.tif')
        achados = destino or origem
        assert len(achados) == 1, f'{prefixo}: {len(achados)} arquivos (o esperado é 1; mais de um quer dizer que o Earth Engine dividiu a exportação). Pare e me mande esta saída.'
        bruto = achados[0]
        if not destino:
            novo = f'{DESTINO}/{os.path.basename(bruto)}'
            shutil.move(bruto, novo)
            bruto = esperar(novo)
        # O Earth Engine corta a exportação na área da imagem, arredondada para fora na grade de 10 m, e não na
        # região pedida. O arquivo bruto pode ter outro tamanho, mas tem de estar na mesma malha de pixels. Ele é
        # colocado na grade inteira do granulo (10980 x 10980) por deslocamento de pixels inteiros, sem reamostrar:
        # o que falta vira 0 (sem dado), 255 na probabilidade de nuvem, e o que passa da grade é cortado.
        # Um arquivo já colocado na grade numa rodada anterior não é regravado; o SHA-256 é conferido no fim.
        canon_drive = f'{CANON}/{prefixo}.tif'
        canon = f'{LOCAL}/{prefixo}.tif'
        print(f'{prefixo}: começando ({"já existe no Drive" if os.path.exists(canon_drive) else "grade inteira ainda não existe no Drive"}).')
        sha_bruto = sha256_arquivo(bruto)
        if (tile, nome, 'bruto') in antigos:
            assert sha_bruto == antigos[(tile, nome, 'bruto')], f'{prefixo}: o arquivo bruto difere do registrado no repositório. Pare e me mande esta saída.'
        refazer = False
        if os.path.exists(canon_drive):
            shutil.copyfile(canon_drive, canon)
        if os.path.exists(canon) and (tile, nome, 'canonico') in antigos:
            sha_antigo = sha256_arquivo(canon)
            if sha_antigo != antigos[(tile, nome, 'canonico')]:
                print(f'{prefixo}: o arquivo da grade inteira no Drive difere do registro (SHA-256 {sha_antigo[:12]}, registrado {antigos[(tile, nome, "canonico")][:12]}). '
                      f'O bruto confere; refaço a grade inteira a partir dele.')
                REPAROS.append({'arquivo': f'canonico/{prefixo}.tif', 'sha256_encontrado': sha_antigo, 'bytes_encontrados': os.path.getsize(canon),
                                'sha256_registrado': antigos[(tile, nome, 'canonico')]})
                os.remove(canon)
                refazer = True
        with rasterio.open(bruto) as s:
            perfil = {'height': s.height, 'width': s.width, 'count': s.count, 'dtype': s.dtypes[0],
                      'crs': s.crs.to_string(), 'transform': tuple(s.transform)[:6]}
            col_off, row_off = S.deslocamento_na_grade(nome, perfil, GRADES[tile]['origem'], tile)
            x0, y0 = GRADES[tile]['origem']
            novo_perfil = {'driver': 'GTiff', 'height': S.LADO, 'width': S.LADO, 'count': s.count, 'dtype': s.dtypes[0],
                           'crs': s.crs, 'transform': rasterio.Affine(10.0, 0.0, x0, 0.0, -10.0, y0),
                           'compress': 'deflate', 'predictor': 2, 'tiled': True, 'blockxsize': 512, 'blockysize': 512}
            if not os.path.exists(canon):
                with rasterio.open(canon + '.parcial.tif', 'w', **novo_perfil) as o:
                    for b in range(1, s.count + 1):
                        o.write(S.colocar_na_grade(s.read(b)[None], col_off, row_off, PREENCHE[nome])[0], b)
                os.replace(canon + '.parcial.tif', canon)
        with rasterio.open(canon) as s:
            perfil_c = {'height': s.height, 'width': s.width, 'count': s.count, 'dtype': s.dtypes[0],
                        'crs': s.crs.to_string(), 'transform': tuple(s.transform)[:6]}
            if nome == 's2':
                S.conferir_grade(tile, perfil_c, GRADES[tile]['origem'])
                grade = perfil_c
                nulos = total = 0
                tem_dado = np.zeros(S.LADO, dtype=bool)
                medias = np.zeros(len(S.BANDAS)); validos = 0
                for r0 in range(0, s.height, 1024):
                    bloco = s.read(window=rasterio.windows.Window(0, r0, s.width, min(1024, s.height - r0))).astype(np.float64)
                    tudo_zero = (bloco == 0).all(axis=0)
                    nulos += int(tudo_zero.sum()); total += tudo_zero.size
                    tem_dado |= (~tudo_zero).any(axis=0)
                    medias += bloco[:, ~tudo_zero].sum(axis=1); validos += int((~tudo_zero).sum())
                zeros = nulos / total
                if tile in ZEROS_REGISTRADOS:
                    assert abs(zeros - ZEROS_REGISTRADOS[tile]) < 1e-12, (
                        f'{tile}: fração de pixels sem dado {zeros} difere da registrada {ZEROS_REGISTRADOS[tile]}. Pare e me mande esta saída.')
                cols = np.flatnonzero(tem_dado)
                if cols.size == 0:
                    with rasterio.open(bruto) as r:
                        com_dado = sum(int((r.read(window=rasterio.windows.Window(0, r0, r.width, min(1024, r.height - r0))) != 0).any(axis=0).sum())
                                       for r0 in range(0, r.height, 1024))
                    raise ValueError(
                        f'{tile}: a grade inteira do s2 tem as 6 bandas em 0 em todos os pixels. Arquivo do Earth Engine {os.path.basename(bruto)}: '
                        f'{perfil["height"]} x {perfil["width"]}, {com_dado} pixels com dado. Grade local {os.path.getsize(canon)} bytes; no Drive '
                        f'{os.path.getsize(canon_drive) if os.path.exists(canon_drive) else "não existe"} bytes. '
                        f'Propriedades do granulo: {c["propriedades"]}. Pare e me mande esta saída.')
                cobertura = {'primeira_coluna_com_dado': int(cols[0]), 'ultima_coluna_com_dado': int(cols[-1])}
                medias = medias / max(validos, 1)
                print(f'{tile}: {zeros:.4f} dos pixels com as 6 bandas em 0; colunas com dado {cols[0]} a {cols[-1]} de {S.LADO}; '
                      f'médias dos válidos', dict(zip(S.BANDAS, np.round(medias, 1))))
            else:
                assert (s.height, s.width, s.count, s.dtypes[0]) == (S.LADO, S.LADO, 1, 'uint8'), (nome, perfil_c)
                assert tuple(s.transform)[:6] == tuple(grade['transform'])
        print(f'{prefixo}: arquivo do Earth Engine {perfil["height"]} x {perfil["width"]}, deslocamento (coluna {col_off}, linha {row_off}) na grade.')
        sha_c = sha256_arquivo(canon)
        arquivos[nome] = {'bruto': {'arquivo': os.path.basename(bruto), 'bytes': os.path.getsize(bruto), 'sha256': sha_bruto,
                                    'altura': perfil['height'], 'largura': perfil['width'], 'transform': list(perfil['transform'])},
                          'canonico': {'arquivo': f'canonico/{prefixo}.tif', 'bytes': os.path.getsize(canon), 'sha256': sha_c},
                          'deslocamento_pixels': {'coluna': col_off, 'linha': row_off}}
        LINHAS.append([os.path.basename(bruto), 'bruto', arquivos[nome]['bruto']['bytes'], arquivos[nome]['bruto']['sha256']])
        LINHAS.append([f'canonico/{prefixo}.tif', 'canonico', arquivos[nome]['canonico']['bytes'], arquivos[nome]['canonico']['sha256']])
        if refazer or not os.path.exists(canon_drive):
            shutil.copyfile(canon, canon_drive)
        esperar(canon_drive)
        assert sha256_arquivo(canon_drive) == sha_c, f'{prefixo}: a cópia no Drive difere do arquivo local. Pare e me mande esta saída.'
    ENTRADAS.append(S.registro_entrada(
        tile, c, c['propriedades'], COLECOES, arquivos,
        {'crs': grade['crs'], 'transform': list(grade['transform']), 'altura': grade['height'], 'largura': grade['width'],
         'cobertura_s2': cobertura},
        zeros, DESLOC[tile]))

# Os arquivos brutos já registrados têm de ser os mesmos, byte a byte (conferido acima). O canônico também, exceto o que
# foi refeito nesta rodada (REPAROS), que fica documentado no registro.
refeitos = {r['arquivo'] for r in REPAROS}
conferidos = 0
for e in ENTRADAS:
    for nome, a in e['arquivos'].items():
        for tipo in ('bruto', 'canonico'):
            if (e['tile'], nome, tipo) in antigos:
                if tipo == 'canonico' and a[tipo]['arquivo'] in refeitos:
                    igual = a[tipo]['sha256'] == antigos[(e['tile'], nome, tipo)]
                    print(f"{e['tile']} {nome}: canônico refeito, SHA-256 {'IGUAL' if igual else 'DIFERENTE'} ao registrado.")
                    continue
                assert a[tipo]['sha256'] == antigos[(e['tile'], nome, tipo)], (
                    f"{e['tile']} {nome} {tipo}: SHA-256 diferente do registrado no repositório. Pare e me mande esta saída.")
                conferidos += 1
print(f'{conferidos} arquivos de rodadas anteriores conferem com o registro do repositório; {len(REPAROS)} refeitos.')

# Conferência final contra o que o Drive guardou de verdade: desmonta (grava tudo), monta de novo e relê os SHA-256.
print('Gravando o Drive e conferindo de novo; pode levar alguns minutos.')
drive.flush_and_unmount()
time.sleep(10)
drive.mount('/content/drive', force_remount=True)
for nome_arq, tipo, _, sha in LINHAS:
    caminho = f'{DESTINO}/{nome_arq}'
    assert os.path.exists(caminho) and sha256_arquivo(caminho) == sha, f'{nome_arq}: o Drive não guardou o arquivo como esperado. Pare e me mande esta saída.'
print('Todos os arquivos do Drive conferem depois de remontar.')

os.makedirs(SAIDA_DRIVE, exist_ok=True)
INFO = {'plano': 'prereg/plano_rq4.md, secao 1', 'commit_repositorio': commit, 's2_exportacao_sha256': SHA_MODULO,
        'reparos_da_grade_inteira': REPAROS, 'data_sensing_utc': S.DATA, 'candidatos_no_dia': TODOS, 'entradas': ENTRADAS,
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rq4_entradas.json', 'w'), indent=2, ensure_ascii=False)
with open(f'{SAIDA_DRIVE}/rq4_entradas_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['arquivo', 'tipo', 'bytes', 'sha256']); w.writerows(LINHAS)
print(open(f'{SAIDA_DRIVE}/rq4_entradas_sha256.csv').read())
print(f'Registro em {SAIDA_DRIVE}. Nenhum modelo foi rodado.')